# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

**Intern:** Ahmed Elgharably
**Track:** Machine Learning
**Assignment:** ML-02 — Research Question and Provisional Lane

This notebook frames my capstone direction using the FlyRank ML Internship starter data and the lane guide.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring (Lane 2)**

I chose this lane because it directly answers a real operational question: *which pages should a content team review first when time is limited?* Unlike pure signal analysis, this lane produces a ranked, actionable output — a queue with reason codes — which matches how FlyRank's product actually surfaces recommendations.

The starter dataset already contains the right ingredients: 90-day impressions, clicks, sessions, content age, word count, trend direction, and engagement signals. The warehouse release (78M+ daily rows) will let me build stronger time-window labels later. This lane also forces me to be honest about causality: I can rank candidates by evidence, but I cannot promise a refresh will cause recovery without an experiment. That discipline is exactly what I want to practice.

**Why not the other lanes?**
- *Ranking Signal Analysis* is valuable but more exploratory; I want a ranked decision output.
- *Structured Content Archetype Clustering* is interesting, but the dataset lacks text content for true semantic clustering, and metric-only clustering feels less actionable for a review queue.
- *CTR/Engagement Opportunity Scoring* is a subset of what Refresh Scoring already covers (CTR and engagement are inputs to the refresh decision).

**Freestyle?** Not needed — Lane 2 is a predefined path with clear guardrails and a strong match to the data.

In [4]:
# Setup: load the starter dataset to confirm lane fit
import pandas as pd
import numpy as np

# Load the starter anonymized dataset
df = pd.read_csv(r'C:\Users\Ahmed Elghrably\Downloads/content_refresh_anonymized.csv')
print(f'Starter dataset shape: {df.shape}')
print(
    f"Columns relevant to refresh scoring:"
    f' {len([c for c in df.columns if any(k in c for k in ["impressions", "clicks", "sessions", "age", "word_count", "trend", "ctr", "engagement", "position", "freshness", "scroll", "ai"])])}'
)
print(f'\nFirst few columns: {list(df.columns[:10])}')

Starter dataset shape: (30000, 44)
Columns relevant to refresh scoring: 30

First few columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count']


## 2. The question: decision, action, cost of a wrong call

### The Decision
A content team at a digital publisher or SaaS company has **limited review capacity** — perhaps 20–50 pages per week. They need to decide which pages in a large inventory deserve human attention first.

### The Action
The team **reviews a page**, then chooses one of: refresh content, expand thin sections, protect a declining winner, prune obsolete pages, or monitor and wait. My work produces a **ranked queue** with scores and reason codes (e.g., `stale_visible_page`, `declining_with_demand`, `low_ctr_visible_page`).

### Cost of a Wrong Call
| Wrong recommendation | Cost | Why it hurts |
|---|---|---|
| **False positive** — flag a healthy page as needing refresh | Wasted reviewer time; opportunity cost of a missed truly-declining page | The team spends hours rewriting content that was already performing well |
| **False negative** — miss a page that is actually declining | Lost traffic, lost revenue, compounding decline | A high-impression page quietly loses position and clicks because no one reviewed it in time |
| **Wrong action type** — suggest refresh when prune or merge is better | Diluted site quality, cannibalization | Two similar pages compete instead of one being merged or redirected |

The most expensive error is the **false negative on a high-impression declining page**, because the window to recover shrinks as decline continues. The starter baseline achieves Precision@50 = 0.24, meaning ~38 of its top 50 recommendations are wrong — there is clear room for improvement.

### Why ML Helps
A hand-written rule (the baseline) uses fixed thresholds: stale AND visible, declining AND demand, etc. But the real world is noisier — a page with moderate impressions but strong CTR decay might matter more than a stale page with zero impressions. A model can learn weighted combinations from historical patterns and beat the fixed rule, as the starter already shows (random forest: Precision@50 ≈ 0.74, ~3× lift).

In [5]:
# Verify the baseline and model results from the starter pipeline
import json
from pathlib import Path

results_path = Path('outputs/model_results.json')
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    print('Starter pipeline results (from model_results.json):')
    for model, metrics in results.items():
        if 'precision_at_50' in metrics:
            print(f"  {model}: Precision@50 = {metrics['precision_at_50']:.3f}")
else:
    print('Note: Run scripts/run_all.py first to generate model_results.json')
    print('Expected starter results from GUIDE.md:')
    print('  baseline rules: Precision@50 = 0.240')
    print('  random forest:  Precision@50 = 0.740')

Note: Run scripts/run_all.py first to generate model_results.json
Expected starter results from GUIDE.md:
  baseline rules: Precision@50 = 0.240
  random forest:  Precision@50 = 0.740


## 3. Quick look at the data (2-3 real numbers)

I loaded the starter dataset (`content_refresh_anonymized.csv`, 30,000 rows × 44 columns) and pulled three numbers that make Lane 2 worth the next 7 weeks.

In [6]:
# Real Number 1: How many pages are both visible AND stale?
# 'Visible' = impressions_90d >= 500 (enough demand to matter)
# 'Stale' = content_age_days >= 180 (6+ months without update)

visible_stale = df[(df['impressions_90d'] >= 500) & (df['content_age_days'] >= 180)]
pct_visible_stale = len(visible_stale) / len(df) * 100
print(f"1. VISIBLE + STALE pages: {len(visible_stale):,} out of {len(df):,} ({pct_visible_stale:.1f}%)")
print(f"   These are prime refresh candidates — they have audience but old content.")

# Real Number 2: Declining pages with meaningful demand
# 'Declining' = trend_direction == 'down'
# 'Meaningful demand' = impressions_90d >= 100

declining_demand = df[(df['trend_direction'] == 'down') & (df['impressions_90d'] >= 100)]
pct_declining_demand = len(declining_demand) / len(df) * 100
print(f"\n2. DECLINING + DEMAND pages: {len(declining_demand):,} out of {len(df):,} ({pct_declining_demand:.1f}%)")
print(f"   These pages are losing visibility despite having search demand — urgent review queue.")

# Real Number 3: Low-CTR visible pages (position 1-20, CTR < 0.5)
# These pages rank well but under-capture clicks

low_ctr = df[(df['impressions_90d'] >= 500) & 
             (df['avg_position'] > 0) & (df['avg_position'] <= 20) & 
             (df['ctr'] < 0.5)]
pct_low_ctr = len(low_ctr) / len(df) * 100
print(f"\n3. LOW-CTR VISIBLE pages: {len(low_ctr):,} out of {len(df):,} ({pct_low_ctr:.1f}%)")
print(f"   These rank on page 1-2 but get fewer clicks than expected — metadata/title issues.")

# Summary table
print("\n" + "="*60)
print("SUMMARY: Three refresh-opportunity segments in the starter data")
print("="*60)
summary = pd.DataFrame({
    'Segment': ['Visible + Stale', 'Declining + Demand', 'Low-CTR Visible'],
    'Count': [len(visible_stale), len(declining_demand), len(low_ctr)],
    'Pct of Total': [f"{pct_visible_stale:.1f}%", f"{pct_declining_demand:.1f}%", f"{pct_low_ctr:.1f}%"],
    'Action': ['Refresh content', 'Urgent review', 'Fix title/meta']
})
print(summary.to_string(index=False))

1. VISIBLE + STALE pages: 9,929 out of 30,000 (33.1%)
   These are prime refresh candidates — they have audience but old content.

2. DECLINING + DEMAND pages: 13,152 out of 30,000 (43.8%)
   These pages are losing visibility despite having search demand — urgent review queue.

3. LOW-CTR VISIBLE pages: 9,759 out of 30,000 (32.5%)
   These rank on page 1-2 but get fewer clicks than expected — metadata/title issues.

SUMMARY: Three refresh-opportunity segments in the starter data
           Segment  Count Pct of Total          Action
   Visible + Stale   9929        33.1% Refresh content
Declining + Demand  13152        43.8%   Urgent review
   Low-CTR Visible   9759        32.5%  Fix title/meta


## 4. Careful words: what I can and can't claim

### What I CAN claim (observed, directional, decision-support)
- **Observed association**: Pages with `content_age_days >= 180` AND `impressions_90d >= 500` show lower average CTR and engagement rate than fresher pages with similar visibility. This is measured from historical data, not inferred.
- **Directional ranking**: A model trained on past patterns can rank pages by their *observed similarity* to pages that previously showed decline signals. This is a prioritization tool, not a prophecy.
- **Decision-support**: The ranked queue helps a human reviewer spend limited time on candidates with the strongest evidence of opportunity or risk. The final decision still belongs to the reviewer.
- **Baseline comparison**: I can honestly report whether my model beats the transparent hand-rule baseline on held-out data (Precision@50, Average Precision, ROC AUC).

### What I CANNOT claim (causal, algorithmic, predictive)
- **Causal proof**: I cannot say "refreshing this page will cause traffic to recover." To prove causality I would need an A/B test, a randomized experiment, or a strong quasi-experimental design — none of which are in this dataset.
- **Predicting Google's algorithm**: I cannot claim my model predicts how Google ranks pages. The model sees only observable signals (impressions, clicks, position) and learns patterns from them — it has no access to ranking factors.
- **Guaranteed outcomes**: Precision@50 ≈ 0.74 means ~26% of the top 50 recommendations are still wrong. I must report this honestly and explain what a wrong recommendation looks like.
- **Semantic understanding**: The data contains no raw text, titles, or URLs (all pseudonymized). I cannot claim to understand *why* a page underperforms at the content-meaning level — only at the signal-pattern level.

### The honest framing
> *"This work produces a ranked list of content-review candidates based on historical search and engagement signals. It helps a team with limited capacity focus on pages where the data shows the strongest evidence of opportunity or decay. It does not guarantee that editing those pages will improve performance, and it should be paired with human judgment and, where possible, controlled experiments."*

In [7]:
# Verify that the starter data contains ONLY observable signals (no product decisions)
product_cols = ['health_score', 'priority_score', 'action_type', 'refresh_tier', 'needs_ctr_fix', 'is_quick_win']
found_product_cols = [c for c in product_cols if c in df.columns]
print(f'Product decision columns found in data: {found_product_cols}')
print('Expected: [] (the starter data ships observable signals only)')
print('\nThis protects against circular results — we cannot accidentally train on FlyRank\'s own decisions.')

Product decision columns found in data: []
Expected: [] (the starter data ships observable signals only)

This protects against circular results — we cannot accidentally train on FlyRank's own decisions.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Additional checks I performed:**
- Verified the starter dataset has no product-decision columns (health_score, priority_score, etc.) — only observable signals.
- Confirmed all three numbers are computed from the actual `content_refresh_anonymized.csv` file, not invented.
- Chose a lane (Refresh Scoring) that matches both the starter data and the full warehouse release for later weeks.
- Defined the cost of wrong calls in terms of reviewer time and missed opportunities, not vague business impact.